In [51]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO

options = Options()
options.headless = True  
driver = webdriver.Chrome(options=options)  

In [52]:
standings_url = "https://fbref.com/en/comps/30/2024-2025/2024-2025-Russian-Premier-League-Stats"

In [53]:
driver.get(standings_url)
time.sleep(2)
data = driver.page_source

In [54]:
soup = BeautifulSoup(data, 'html.parser')
standings_table = soup.select('table.stats_table')[0]
links = standings_table.find_all('a')
links = [l.get("href") for l in links]
links = [l for l in links if '/squads/' in l]

In [55]:
team_urls = [f"https://fbref.com{l}" for l in links]

print(team_urls)

['https://fbref.com/en/squads/fa11a9cc/2024-2025/Krasnodar-Stats', 'https://fbref.com/en/squads/98ce363d/2024-2025/Zenit-Stats', 'https://fbref.com/en/squads/f0c0c2c2/2024-2025/CSKA-Moscow-Stats', 'https://fbref.com/en/squads/8c635914/2024-2025/Spartak-Moscow-Stats', 'https://fbref.com/en/squads/541a280b/2024-2025/Dynamo-Moscow-Stats', 'https://fbref.com/en/squads/5a8dc328/2024-2025/Lokomotiv-Moscow-Stats', 'https://fbref.com/en/squads/5625a7da/2024-2025/Rubin-Kazan-Stats', 'https://fbref.com/en/squads/d60423ef/2024-2025/Rostov-Stats', 'https://fbref.com/en/squads/9e9e1971/2024-2025/FK-Akron-Tolyatti-Stats', 'https://fbref.com/en/squads/483ffd93/2024-2025/Samara-Stats', 'https://fbref.com/en/squads/f7823485/2024-2025/Dynamo-Makhachkala-Stats', 'https://fbref.com/en/squads/224b0274/FC-Khimki-Stats', 'https://fbref.com/en/squads/c28444cc/2024-2025/Nizhny-Novgorod-Stats', 'https://fbref.com/en/squads/8aa1135c/2024-2025/Akhmat-Grozny-Stats', 'https://fbref.com/en/squads/555a9123/2024-2025/

In [56]:
driver.get(team_urls[0])
time.sleep(2)
data = driver.page_source

In [57]:
matches = pd.read_html(StringIO(data), match="Scores & Fixtures")[0]

In [58]:
soup = BeautifulSoup(data, 'html.parser')
links = soup.find_all('a')
links = [l.get("href") for l in links]
links = [l for l in links if l and 'all_comps/shooting/' in l]

In [59]:
driver.get(f"https://fbref.com{links[0]}")
time.sleep(2)
data = driver.page_source

In [60]:
shooting = pd.read_html(StringIO(data), match="Shooting")[0]

In [61]:
shooting.head()

For Krasnodar                                                                \
           Date   Time                    Comp        Round  Day Venue Result   
0    2024-07-21  20:00  Russian Premier League  Matchweek 1  Sun  Away      D   
1    2024-07-28  20:00  Russian Premier League  Matchweek 2  Sun  Home      D   
2    2024-08-04  17:30  Russian Premier League  Matchweek 3  Sun  Away      D   
3    2024-08-10  20:00  Russian Premier League  Matchweek 4  Sat  Home      W   
4    2024-08-18  20:00  Russian Premier League  Matchweek 5  Sun  Home      W   

                              Standard                                    \
    GF   GA          Opponent      Gls  Sh SoT  SoT%  G/Sh G/SoT Dist PK   
0  1.0  1.0     Akhmat Grozny        1  12   3  25.0  0.08  0.33  NaN  0   
1  0.0  0.0  D'mo Makhachkala        0  18   3  16.7  0.00  0.00  NaN  0   
2  0.0  0.0    Fakel Voronezh        0  13   2  15.4  0.00  0.00  NaN  0   
3  2.0  1.0       CSKA Moscow        2  16   3  18.8  0.13  0.67  NaN  0   
4  2.0  1.0   Nizhny Novgorod        2  19   4  21.1  0.11  0.50  NaN  0   

        Unnamed: 19_level_0  
  PKatt        Match Report  
0     0        Match Report  
1     0        Match Report  
2     0        Match Report  
3     0        Match Report  
4     0        Match Report

In [62]:
shooting.columns = shooting.columns.droplevel()

In [64]:
team_data = matches.merge(shooting[["Date", "Sh", "SoT", "Dist", "PK", "PKatt"]], on="Date")

In [65]:
team_data.head()

,Date,Time,Comp,Round,Day,Venue,Result,GF,GA,Opponent,...,Formation,Opp Formation,Referee,Match Report,Notes,Sh,SoT,Dist,PK,PKatt
0,2024-07-21,20:00,Russian Premier League,Matchweek 1,Sun,Away,D,1,1,Akhmat Grozny,...,3-4-3,5-3-2,Vitaly Meshkov,Match Report,NaN,12,3,NaN,0,0
1,2024-07-28,20:00,Russian Premier League,Matchweek 2,Sun,Home,D,0,0,D'mo Makhachkala,...,4-2-3-1,3-4-3,Pavel Shadykhanov,Match Report,NaN,18,3,NaN,0,0
2,2024-08-04,17:30,Russian Premier League,Matchweek 3,Sun,Away,D,0,0,Fakel Voronezh,...,4-2-3-1,3-4-3,Aleksey Sukhoy,Match Report,NaN,13,2,NaN,0,0
3,2024-08-10,20:00,Russian Premier League,Matchweek 4,Sat,Home,W,2,1,CSKA Moscow,...,4-2-3-1,3-4-3,Vladimir Moskalev,Match Report,NaN,16,3,NaN,0,0
4,2024-08-18,20:00,Russian Premier League,Matchweek 5,Sun,Home,W,2,1,Nizhny Novgorod,...,4-2-3-1,5-4-1,Artem Chistyakov,Match Report,NaN,19,4,NaN,0,0


In [72]:
years = list(range(2025, 2018, -1))
all_matches = []

In [73]:
standings_url = "https://fbref.com/en/comps/30/Russian-Premier-League-Stats"

In [74]:
for year in years:
    driver.get(standings_url)
    time.sleep(2)
    data = driver.page_source
    soup = BeautifulSoup(data, 'html.parser')
    standings_table = soup.select('table.stats_table')[0]

    links = [l.get("href") for l in standings_table.find_all('a')]
    links = [l for l in links if '/squads/' in l]
    team_urls = [f"https://fbref.com{l}" for l in links]
    
    previous_season = soup.select("a.prev")[0].get("href")
    standings_url = f"https://fbref.com{previous_season}"
    
    for team_url in team_urls:
        team_name = team_url.split("/")[-1].replace("-Stats", "").replace("-", " ")
        driver.get(team_url)
        time.sleep(2)
        data = driver.page_source
        matches = pd.read_html(StringIO(data), match="Scores & Fixtures")[0]
        soup = BeautifulSoup(data, 'html.parser')
        links = [l.get("href") for l in soup.find_all('a')]
        links = [l for l in links if l and 'all_comps/shooting/' in l]
        driver.get(f"https://fbref.com{links[0]}")
        time.sleep(2)
        data = driver.page_source
        shooting = pd.read_html(StringIO(data), match="Shooting")[0]
        shooting.columns = shooting.columns.droplevel()
        try:
            team_data = matches.merge(shooting[["Date", "Sh", "SoT", "Dist", "PK", "PKatt"]], on="Date")
        except ValueError:
            continue
        team_data = team_data[team_data["Comp"] == "Premier League"]
        
        team_data["Season"] = year
        team_data["Team"] = team_name
        all_matches.append(team_data)
        time.sleep(3)  

In [75]:
len(all_matches)

112

In [76]:
match_df = pd.concat(all_matches)

In [77]:
match_df.columns = [c.lower() for c in match_df.columns]

In [79]:
match_df.isna().sum()

date                0
time                0
comp                0
round               0
day                 0
venue               0
result              0
gf                  0
ga                  0
opponent            0
poss               14
attendance        116
captain             2
formation           0
opp formation       0
referee             0
match report        0
notes            2386
sh                  0
sot                 0
dist             2388
pk                  0
pkatt               0
season              0
team                0
xg               2388
xga              2388
dtype: int64

In [80]:
match_df.to_csv("matches.csv", index=False)